In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from src.config import CACHE

features = pd.read_pickle(CACHE / "features_deriv_4s.pkl")

# keep only AD and CN subjects
data = features[features.label.isin(["AD", "CN"])].reset_index(drop=True)
X = data.drop(columns=["subject", "label"]).values      # the 114 features
y = (data.label == "AD").astype(int).values             # AD = 1, CN = 0
groups = data.subject.values                            # who each epoch belongs to

# 5 folds, and all epochs of one subject always stay together
folds = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))

fold_scores = []
for train, test in folds.split(X, y, groups):
    model.fit(X[train], y[train])                                   # learn from training epochs only

    epoch_prob = model.predict_proba(X[test])[:, 1]                 # chance of AD for each test epoch
    table = pd.DataFrame({"subject": groups[test], "prob": epoch_prob, "true": y[test]})
    per_subject = table.groupby("subject").mean()                   # average over each subject's epochs

    guess = (per_subject.prob > 0.5).astype(int)                    # AD if average chance is over 50%
    fold_scores.append((guess == per_subject.true.round()).mean())  # fraction of subjects correct

print("accuracy per fold:", np.round(fold_scores, 2))
print("mean:", round(np.mean(fold_scores), 2), "+/-", round(np.std(fold_scores), 2))

accuracy per fold: [0.85 0.69 0.79 0.92 0.77]
mean: 0.8 +/- 0.08


In [2]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from src.evaluate import subject_cv

def make_logreg():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))

accuracies = []
for seed in range(10):                                   # 10 different shuffles
    res = subject_cv(features, make_logreg, ["AD", "CN"], seed)
    accuracies.append((res.true == res.guess).mean())    # fraction of all 65 subjects correct

print(np.round(accuracies, 2))
print("mean:", round(np.mean(accuracies), 3), "+/-", round(np.std(accuracies), 3))

[0.77 0.77 0.83 0.78 0.78 0.82 0.78 0.8  0.8  0.8 ]
mean: 0.794 +/- 0.018


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

def make_rf():
    return RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=42)

models = {"LogReg": make_logreg, "RandomForest": make_rf}
tasks = [["AD", "CN"], ["AD", "FTD", "CN"]]

for classes in tasks:
    for name, make_model in models.items():
        accuracies = []
        for seed in range(5):                                    # 5 shuffles, to keep it quick
            res = subject_cv(features, make_model, classes, seed)
            accuracies.append((res.true == res.guess).mean())
        print("-".join(classes), "|", name, "| mean accuracy:", round(np.mean(accuracies), 3))
        print(confusion_matrix(res.true, res.guess))             # last shuffle: rows = true, columns = guessed
        print()

AD-CN | LogReg | mean accuracy: 0.788
[[28  8]
 [ 6 23]]

AD-CN | RandomForest | mean accuracy: 0.831
[[30  6]
 [ 4 25]]

AD-FTD-CN | LogReg | mean accuracy: 0.564
[[20 12  4]
 [10  6  7]
 [ 2  5 22]]

AD-FTD-CN | RandomForest | mean accuracy: 0.645
[[28  1  7]
 [16  5  2]
 [ 4  2 23]]



In [ ]:
from src.evaluate import repeat_cv
from src.config import RESULTS

rows = []
for classes in tasks:
    for name, make_model in models.items():
        row = repeat_cv(features, make_model, classes)
        row.update({"task": "-".join(classes), "model": name})
        rows.append(row)

table = pd.DataFrame(rows).round(3)
print(table)
table.to_csv(RESULTS / "baseline_results.csv", index=False)